# Lesson 2.5: LangChain - Building a RAG App from Parts
### Guided exercise (assignment)

You run the RAG chain of the demo on **your 8 questions** (T-questions), change **one** setting, compare it with
the same RAG written without LangChain chains, and explain the errors. Start in class, finish at home (about
45-60 min), **submit before Lesson 2.6**.

| Part | What you do | Deliverable |
|---|---|---|
| A | Predict, then run the chain on 8 questions | prediction + table (CSV) |
| B | Change ONE setting (k, filter, a rule, the gate) and re-run | before/after table |
| C | One question: hand-made RAG vs LangChain | the two prompts + a comparison paragraph |
| D | Read the errors: retrieval vs generation; does the source support the claim? | 2 analyses + 3 support checks |

**Rules:** do not edit cell S1 or the questions; write predictions **before** running; do not type outputs by
hand; quote the chunks and answers of YOUR run.

**How to run**
- *Local (recommended) - conda env `vti`* (the same env as Lessons F1-2.4). Today needs **new packages**:
  LangChain. In a terminal (Anaconda Prompt on Windows), from this folder:
  ```bash
  conda activate vti                 # first time ever: conda create -n vti python=3.12 ipython
  pip install -r requirements.txt    # adds langchain, langchain-huggingface, langchain-chroma
  ```
  Then run `jupyter lab` from the activated env, or open this notebook in VS Code and choose the **vti** kernel.
  The chat model is Qwen2.5-1.5B-Instruct of Lessons 2.1-2.3 (~3.1 GB) and the embedding model of 2.3-2.4
  (~0.5 GB): both are already on your laptop if you did those lessons.
- *Google Colab:* `File -> Upload notebook`, then in a new first cell run
  `!pip install -q "langchain>=1.4,<2" "langchain-huggingface>=1.2,<2" "langchain-chroma>=1.1,<2" "chromadb>=1.5"
  sentence-transformers` and restart the session. A free CPU runtime works (slower); the data is inside cell S1.
- Run the cells from top to bottom (`Shift + Enter`). Each model call takes 5-15 s on a laptop CPU.

## S0 - Environment check

In [2]:
# Cell S0 - environment check (run it, no changes needed)
import os, sys, json
from importlib.metadata import version
import torch, transformers, chromadb, langchain, langchain_core

print("Python       ", sys.version.split()[0], "| env:", os.environ.get("CONDA_DEFAULT_ENV") or sys.prefix)
print("PyTorch      ", torch.__version__, "| GPU:", torch.cuda.is_available(), "(no GPU needed)")
print("Transformers ", transformers.__version__, "| Chroma", chromadb.__version__)
print("LangChain    ", langchain.__version__, "| core", langchain_core.__version__,
      "| langchain-huggingface", version("langchain-huggingface"), "| langchain-chroma", version("langchain-chroma"))
assert sys.version_info >= (3, 10), "Python 3.10 or newer is required"
assert int(langchain.__version__.split(".")[0]) >= 1, "LangChain 1.x is required: pip install -U langchain"
print("[OK] Environment ready")

Python        3.12.15 | env: vti
PyTorch       2.14.1+cpu | GPU: False (no GPU needed)
Transformers  5.19.0 | Chroma 1.5.9
LangChain     1.4.7 | core 1.7.0 | langchain-huggingface 1.2.3 | langchain-chroma 1.1.0
[OK] Environment ready


## S1 - Provided helpers

The cell below contains the data and support code. **You don't need to read it** - just know what goes in and out:

| Helper | Input -> output |
|---|---|
| `KB_DOCS`, `QUESTIONS`, `TASK_QUESTIONS`, `TICKETS` | the Company X handbook (today's version); 8 + 8 questions with the correct `source_id` and expected facts; 5 tickets of Lesson 2.3 with ground truth |
| `load_chat_model()` | the local Qwen model **as a LangChain chat model** (`LocalChatModel`): `chat.invoke(messages)` -> an `AIMessage` |
| `show_prompt()` | the text the model **really** received on the last call (after LangChain and the chat template), and its raw answer |
| `load_embeddings()`, `build_vector_store(embeddings)` | the embedding model; the Lesson 2.4 KB (fixed chunks 60/15) as a LangChain `Chroma` vector store |
| `show_docs(docs)`, `number_docs(docs)` | print retrieved chunks; turn them into the context text with labels [1], [2], [3] |
| `make_rag(retriever, generate, gate=)` | wire the RAG chain; `gate=True`: no chunk above the threshold -> code says "not found" |
| `show_rag(out)`, `check_rag(question, docs, answer)` | print a result (chunks first, then the answer); score it: retrieval / enough_info / sources / fact |
| `run_questions(rag, questions)`, `compare_runs(a, b)` | run all questions and count retrieval vs generation errors; two runs side by side |
| `check_ticket(value, id)`, `manual_rag(question, chat, store)` | Part 2: check a ticket; the same RAG without LangChain chains |
| `save_json(value, path)` / `save_results(rows, path)` | save a JSON file / a CSV table (opens in Excel) |

In [3]:
# Cell S1 - load the helpers and the data (run it; no need to read or edit)
# Helper tools for Lesson 2.5 - run this cell; you do NOT need to read or edit it.
# Contains: the Company X data (handbook, questions, tickets), a LangChain chat model that runs the local Qwen model
# (LocalChatModel), the knowledge base of Lesson 2.4 as a LangChain vector store, tools to print the prompt the
# model really received, and checks for a RAG answer (retrieval error vs generation error). Checks print [OK] / [NO].
import csv
import json
import os
import re
import time
import uuid
from pathlib import Path
from typing import Any

import torch
from langchain_core.documents import Document
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.outputs import ChatGeneration, ChatResult
from langchain_core.utils.function_calling import convert_to_openai_tool
from transformers import AutoModelForCausalLM, AutoTokenizer

# ---------- Company X data (fictional), from Lessons 2.3-2.4 ----------
# KB_DOCS: the handbook as it is today (9 documents with sections; the returns policy is version 2).
KB_DOCS = [{'source_id': 'POL-RET-01',
  'title': 'Returns',
  'category': 'returns',
  'updated_at': '2026-10-10',
  'version': 2,
  'text': '## Change-of-mind returns\n'
          'You can return an item within 14 days of delivery if you changed your mind. The item must be unused, '
          'clean and in\n'
          'its original box with all accessories and the warranty card.\n'
          '## Exchanges\n'
          'If you want another colour or model instead, choose "exchange" when you book the return. We ship the '
          'new item\n'
          'when the old one reaches our warehouse. If the new item costs more, you pay the difference; if it costs '
          'less, we\n'
          'refund the difference.\n'
          '## Items that cannot be returned\n'
          'Filters, vacuum bags and other parts that touch food or dust cannot be returned once the packet is '
          'open. Items\n'
          'sold as "final sale" in the shop cannot be returned either, unless they are faulty.\n'
          '## Pickup and refund\n'
          'Book the return in your account and a courier collects the item from your address. From 10 October 2026 '
          'return\n'
          'shipping is free. The refund goes back to the original payment method within 7 working days after the '
          'item\n'
          'reaches our warehouse.'},
 {'source_id': 'POL-DMG-01',
  'title': 'Damaged or faulty items',
  'category': 'damaged',
  'updated_at': '2026-09-01',
  'version': 1,
  'text': '## Check the box at delivery\n'
          'Look at the box in front of the driver. If it is crushed, wet or open, take a photo and write it on the '
          'delivery\n'
          'note, or refuse the parcel: we send a new one at no cost.\n'
          '## Report the problem\n'
          'If an item arrives damaged or stops working, report it within 7 days of delivery in your account or by '
          'phone.\n'
          'Add a photo or a short video of the problem if you can.\n'
          '## Missing parts\n'
          'If a part or an accessory is missing from the box, report it within 7 days. We send the missing part by '
          'express\n'
          'delivery, free of charge, usually within 3 working days.\n'
          '## Replacement or refund\n'
          'Company X collects the item free of charge. We ship a replacement within 5 working days after the item '
          'reaches\n'
          'our warehouse. If you prefer, we refund the full price instead, including the delivery fee.\n'
          '## After 7 days\n'
          "Problems reported more than 7 days after delivery go to the manufacturer's warranty centre. We send you "
          'the\n'
          "centre's address and help you book the repair."},
 {'source_id': 'POL-WAR-01',
  'title': 'Warranty',
  'category': 'warranty',
  'updated_at': '2026-08-15',
  'version': 1,
  'text': '## Who repairs the item\n'
          "Every appliance has the manufacturer's warranty. Warranty repairs are done by the manufacturer's "
          'service centre,\n'
          'not by Company X. Keep the warranty card and the invoice: the centre asks for both.\n'
          '## How to make a claim\n'
          'Call the service centre on the warranty card or bring the item to the centre. Most repairs take 7 to 14 '
          'days.\n'
          'For fridges and washing machines a technician comes to your home.\n'
          '## How long the warranty lasts\n'
          'Small kitchen appliances such as kettles, blenders, toasters and rice cookers have 12 months. Fans, '
          'vacuum\n'
          'cleaners and air purifiers have 24 months. Fridges, washing machines and air conditioners have 36 '
          'months.\n'
          '## Extended warranty\n'
          "At checkout you can add 12 more months of warranty for 10% of the item's price. The extended warranty "
          'covers the\n'
          "same faults as the manufacturer's warranty and is handled by Company X.\n"
          '## What is not covered\n'
          'The warranty does not cover damage from drops, water, the wrong voltage or repairs by someone else. '
          'Filters and\n'
          'other parts that wear out are not covered either.'},
 {'source_id': 'POL-DEL-01',
  'title': 'Delivery',
  'category': 'delivery',
  'updated_at': '2026-09-10',
  'version': 1,
  'text': '## Delivery times\n'
          'We deliver nationwide. Ho Chi Minh City and Ha Noi: 1-2 working days. Other provinces: 2-5 working '
          'days. Orders\n'
          'placed after 4 pm count from the next working day.\n'
          '## Delivery fee\n'
          'The delivery fee is 45,000 VND per order. Delivery is free for orders over 1,000,000 VND.\n'
          '## Tracking your order\n'
          'When your order ships you get a text message with a tracking link. The link shows where the parcel is '
          'and the\n'
          'name and phone number of the driver on the day of delivery.\n'
          '## Large appliances\n'
          'Fridges and washing machines are delivered by our own team in a 4-hour time slot that you choose at '
          'checkout.\n'
          'The team carries the item to the room you want and removes the packaging.\n'
          '## Late or missed deliveries\n'
          'If the driver misses your time slot or the order is more than 3 days late, contact support with your '
          'order code.\n'
          'We book a new slot and the driver calls 30 minutes before arrival. You may cancel a late order instead '
          'and get a\n'
          'full refund with no fee.'},
 {'source_id': 'POL-ORD-01',
  'title': 'Changing or cancelling an order',
  'category': 'orders',
  'updated_at': '2026-09-01',
  'version': 1,
  'text': '## Order status\n'
          'Your account shows the status of every order: confirmed, packed, shipped, delivered. You also get a '
          'text message\n'
          'at each step.\n'
          '## Before the order ships\n'
          'The delivery address, the time slot and the payment method can be changed free of charge until the '
          'order ships.\n'
          'An order that has not shipped can be cancelled at no cost in your account.\n'
          '## Orders in several parcels\n'
          'A large order may arrive in several parcels on different days. Each parcel has its own tracking link; '
          'the order\n'
          'shows "delivered" only when the last parcel arrives.\n'
          '## After the order ships\n'
          'After shipping, the order can no longer be changed. Contact the carrier with the tracking number to ask '
          'for a new\n'
          'address, or wait for delivery and use a change-of-mind return.'},
 {'source_id': 'POL-PAY-01',
  'title': 'Payments and refunds',
  'category': 'payment',
  'updated_at': '2026-07-01',
  'version': 1,
  'text': '## Ways to pay\n'
          'We accept bank cards, bank transfer, e-wallets and cash on delivery. Cash on delivery is available for '
          'orders up\n'
          'to 10,000,000 VND.\n'
          '## Monthly instalments\n'
          'Orders over 3,000,000 VND can be paid in 3, 6 or 12 monthly instalments at 0% interest with partner '
          'banks. Choose\n'
          'the instalment plan at checkout; the bank approves it within a few minutes.\n'
          '## Invoices\n'
          'A VAT invoice can be requested at checkout or within the same month. Write the company name, address '
          'and tax\n'
          'code exactly as registered; the invoice is sent by email.\n'
          '## Refund times\n'
          'Refunds go back to the method you paid with: e-wallet within 3 working days, card or bank account '
          'within 7\n'
          'working days. A double charge is refunded in 3-5 working days, depending on your bank.'},
 {'source_id': 'FAQ-AP-01',
  'title': 'Air purifiers AP-20 and AP-40',
  'category': 'product',
  'updated_at': '2026-06-20',
  'version': 1,
  'text': '## Which model for which room\n'
          'The AP-20 is for rooms up to 30 m2 and the AP-40 for rooms up to 45 m2. Both have three fan speeds and '
          'a timer\n'
          'from 1 to 8 hours.\n'
          '## Air quality light\n'
          'A light on the top shows the air quality: blue is good, yellow is fair and red is bad. In automatic '
          'mode the fan\n'
          'speed follows the light.\n'
          '## Filters\n'
          'Each purifier comes with one filter. Filters are not washable: replace them every 6 months. The spare '
          'filter for\n'
          'the AP-20 is HF-2013 (390,000 VND); for the AP-40 it is HF-4013 (450,000 VND).\n'
          '## Cleaning and noise\n'
          'Wipe the outside and the sensor with a dry cloth once a month. In night mode the AP-20 makes 24 dB and '
          'the AP-40\n'
          'makes 28 dB, quieter than a whisper.'},
 {'source_id': 'SVC-INS-01',
  'title': 'Installation service',
  'category': 'service',
  'updated_at': '2026-10-01',
  'version': 1,
  'text': '## What we install\n'
          'Our technicians install washing machines, dishwashers and air conditioners. Fridges and other '
          'appliances need no\n'
          'installation: the delivery team sets them up.\n'
          '## Booking and price\n'
          'Book the installation when you order. It costs 200,000 VND for a washing machine or dishwasher and '
          '500,000 VND\n'
          'for an air conditioner. It takes place within 3 days after delivery.\n'
          '## Before the technician comes\n'
          'Make sure there is a socket and a water tap near the place of the appliance. For an air conditioner, '
          'the outside\n'
          'unit needs a wall or a balcony that the technician can reach safely.\n'
          '## Taking away your old appliance\n'
          'At the same visit the technician can take away your old appliance for free. Tell us when you book, so '
          'that the\n'
          'team brings the right tools and a trolley.'},
 {'source_id': 'PRM-DEL-10',
  'title': 'October delivery offer',
  'category': 'delivery',
  'updated_at': '2026-10-01',
  'version': 1,
  'text': '## Free delivery in October\n'
          'From 1 to 31 October 2026, delivery is free for all orders over 500,000 VND, in every province. Orders '
          'up to\n'
          '500,000 VND pay the normal delivery fee. The offer cannot be combined with other delivery vouchers.'}]

# The 8 demo questions and your 8 test questions: the correct source_id (None = no document answers it)
# and the facts a correct answer must contain (any of them).
QUESTIONS = [{'qid': 'Q01',
  'question': "How many days do I have to send back something I don't like?",
  'source_id': 'POL-RET-01',
  'kind': 'paraphrase',
  'facts': ['14 days']},
 {'qid': 'Q02',
  'question': 'My new kettle arrived broken. What should I do?',
  'source_id': 'POL-DMG-01',
  'kind': 'paraphrase',
  'facts': ['7 days']},
 {'qid': 'Q04',
  'question': 'How much is HF-4013?',
  'source_id': 'FAQ-AP-01',
  'kind': 'product code',
  'facts': ['450,000', '450000']},
 {'qid': 'Q05',
  'question': 'My order is 700,000 VND. Do I pay for delivery?',
  'source_id': 'PRM-DEL-10',
  'kind': 'two versions',
  'facts': ['free', 'do not need to pay', "don't need to pay", 'do not pay', 'no delivery fee']},
 {'qid': 'Q06',
  'question': 'The driver did not come in my time slot. What now?',
  'source_id': 'POL-DEL-01',
  'kind': 'late section',
  'facts': ['new slot', 'cancel', 'contact support']},
 {'qid': 'Q07',
  'question': 'Can the technician take my old washing machine away?',
  'source_id': 'SVC-INS-01',
  'kind': 'late section',
  'facts': ['free', 'away']},
 {'qid': 'Q09',
  'question': 'Do you have a showroom in Da Nang where I can see the products?',
  'source_id': None,
  'kind': 'out of scope',
  'facts': None},
 {'qid': 'Q10',
  'question': 'Do you give a discount to students?',
  'source_id': None,
  'kind': 'out of scope',
  'facts': None}]
TASK_QUESTIONS = [{'qid': 'T01',
  'question': "I opened the box and I don't want it anymore. Can I return it?",
  'source_id': 'POL-RET-01',
  'kind': 'paraphrase',
  'facts': ['14 days']},
 {'qid': 'T02',
  'question': 'Who pays the courier when I send an item back?',
  'source_id': 'POL-RET-01',
  'kind': 'two versions',
  'facts': ['free']},
 {'qid': 'T03',
  'question': 'Is HF-2013 the right filter for my purifier?',
  'source_id': 'FAQ-AP-01',
  'kind': 'product code',
  'facts': ['AP-20']},
 {'qid': 'T04',
  'question': 'I paid twice for the same order. When do I get one payment back?',
  'source_id': 'POL-PAY-01',
  'kind': 'late section',
  'facts': ['3-5 working days', '3 to 5']},
 {'qid': 'T06',
  'question': 'How much does it cost to install an air conditioner?',
  'source_id': 'SVC-INS-01',
  'kind': 'paraphrase',
  'facts': ['500,000', '500000']},
 {'qid': 'T08',
  'question': 'How loud is the bigger purifier at night?',
  'source_id': 'FAQ-AP-01',
  'kind': 'late section',
  'facts': ['28 dB']},
 {'qid': 'T09',
  'question': 'Can I collect my order myself at your warehouse?',
  'source_id': None,
  'kind': 'out of scope',
  'facts': None},
 {'qid': 'T10',
  'question': 'Do you buy back old phones?',
  'source_id': None,
  'kind': 'out of scope',
  'facts': None}]

# The 5 demo tickets of Lesson 2.3 with their ground truth JSON (Part 2: the same job, with LangChain).
TICKETS = {'J01': {'text': 'From: Lan Tran\n'
                 'Hi, I ordered a rice cooker (order #CX-10482) on 2 October. It arrived yesterday but the inner '
                 'pot has a deep scratch and the lid does not close properly. I would like a replacement, not a '
                 'refund. I am at home after 6 pm on weekdays.\n'
                 'Thanks, Lan',
         'truth': {'customer': 'Lan Tran',
                   'order_id': 'CX-10482',
                   'product': 'rice cooker',
                   'category': 'damaged',
                   'requests': ['a replacement'],
                   'amount_vnd': None,
                   'needs_review': False}},
 'J02': {'text': 'From: Minh\n'
                 'Hello, my blender stopped working after two weeks. It makes a burning smell when I turn it on. '
                 'What can you do? This is really disappointing.',
         'truth': {'customer': 'Minh',
                   'order_id': None,
                   'product': 'blender',
                   'category': 'damaged',
                   'requests': ['help with the broken blender'],
                   'amount_vnd': None,
                   'needs_review': True}},
 'J03': {'text': 'From: Nam\n'
                 'Hi, my neighbour told me to write to you. Her order CX-10498 came fast. Mine did not: I ordered '
                 'a standing fan 9 days ago and it is still not here. When will it arrive?\n'
                 'Nam',
         'truth': {'customer': 'Nam',
                   'order_id': None,
                   'product': 'standing fan',
                   'category': 'delivery',
                   'requests': ['the arrival date'],
                   'amount_vnd': None,
                   'needs_review': True}},
 'J04': {'text': 'From: Tuan Le\n'
                 'I was charged twice for my microwave, order #CX-10702: two payments of 2,490,000 VND on the same '
                 'day. Please refund one of them to my card.',
         'truth': {'customer': 'Tuan Le',
                   'order_id': 'CX-10702',
                   'product': 'microwave',
                   'category': 'refund',
                   'requests': ['refund one payment'],
                   'amount_vnd': 2490000,
                   'needs_review': False}},
 'J05': {'text': 'From: Lien\n'
                 'Please cancel my blender order, I found it cheaper in another shop. It has not shipped yet.\n'
                 'Lien',
         'truth': {'customer': 'Lien',
                   'order_id': None,
                   'product': 'blender',
                   'category': 'refund',
                   'requests': ['cancel the order'],
                   'amount_vnd': None,
                   'needs_review': True}}}

MAIN_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"     # chat model of Lessons 2.1-2.3, Apache-2.0, ~3.1 GB download
SMALL_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"    # fallback for weak laptops, ~1 GB, makes more mistakes
EMBED_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"   # embedding model of Lessons 2.3-2.4
KB_CONFIG = {"strategy": "fixed", "size": 60, "overlap": 15, "k": 3, "threshold": 0.35}   # locked in Lesson 2.4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.bfloat16
torch.set_num_threads(max(1, min(4, os.cpu_count() or 1)))
STORE = "kb_store"
LAST = {"prompt": None, "answer": None, "seconds": None}   # the last text sent to the model and its raw answer


# ======================================================================================  the chat model
class LocalChatModel(BaseChatModel):
    """A LangChain chat model that runs a local Hugging Face model on this laptop (no server, no API key).
    It turns LangChain messages into the model's chat template - including the tools, if any are bound - and turns
    the model's <tool_call> blocks back into LangChain tool calls. Greedy decoding: the same input gives the same
    answer. This is what an "integration" such as ChatOpenAI or ChatOllama does for its own model."""
    model_name: str = MAIN_MODEL
    max_new_tokens: int = 300
    tok: Any = None
    model: Any = None

    def __init__(self, **kw):
        super().__init__(**kw)
        t0 = time.time()
        self.tok = AutoTokenizer.from_pretrained(self.model_name)
        self.model = AutoModelForCausalLM.from_pretrained(self.model_name, dtype=DTYPE).to(DEVICE).eval()
        n = sum(p.numel() for p in self.model.parameters())
        print(f"Chat model {self.model_name} | {n / 1e9:.2f} billion parameters | {DEVICE} | "
              f"loaded in {time.time() - t0:.0f} s")

    @property
    def _llm_type(self):
        return "local-transformers-chat"

    @staticmethod
    def _to_dicts(messages):
        out = []
        for m in messages:
            if isinstance(m, SystemMessage):
                out.append({"role": "system", "content": m.content})
            elif isinstance(m, HumanMessage):
                out.append({"role": "user", "content": m.content})
            elif isinstance(m, AIMessage):
                d = {"role": "assistant", "content": m.content or ""}
                if m.tool_calls:
                    d["tool_calls"] = [{"type": "function", "function": {"name": t["name"], "arguments": t["args"]}}
                                       for t in m.tool_calls]
                out.append(d)
            elif isinstance(m, ToolMessage):
                out.append({"role": "tool", "content": str(m.content)})
        return out

    def bind_tools(self, tools, tool_choice=None, **kw):
        """Attach tools. tool_choice = a tool name (or "any"/"required"): the model MUST call a tool. Hosted APIs
        enforce this while decoding; a local model cannot, so we add one visible instruction to the system text."""
        return self.bind(tools=[convert_to_openai_tool(t) for t in tools], tool_choice=tool_choice, **kw)

    def _generate(self, messages, stop=None, run_manager=None, tools=None, tool_choice=None, **kw):
        dicts = self._to_dicts(messages)
        if tools and tool_choice and tool_choice not in ("auto", "none"):
            name = tool_choice if isinstance(tool_choice, str) and tool_choice not in ("any", "required") else \
                (tool_choice.get("function", {}).get("name") if isinstance(tool_choice, dict) else None)
            must = (f"You must answer by calling the function {name} inside <tool_call></tool_call> tags."
                    if name else "You must answer by calling one of the functions inside <tool_call></tool_call> tags.")
            if dicts and dicts[0]["role"] == "system":
                dicts[0] = dict(dicts[0], content=dicts[0]["content"] + "\n" + must)
            else:
                dicts.insert(0, {"role": "system", "content": must})
        LAST["prompt"] = self.tok.apply_chat_template(dicts, tools=tools, add_generation_prompt=True, tokenize=False)
        ids = self.tok.apply_chat_template(dicts, tools=tools, add_generation_prompt=True, return_tensors="pt",
                                           return_dict=True).to(DEVICE)
        t0 = time.time()
        with torch.inference_mode():
            out = self.model.generate(**ids, max_new_tokens=self.max_new_tokens, do_sample=False,
                                      pad_token_id=self.tok.eos_token_id)
        text = self.tok.decode(out[0, ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        LAST.update(answer=text, seconds=round(time.time() - t0, 1))
        calls = []
        for block in re.findall(r"<tool_call>\s*(.*?)\s*</tool_call>", text, flags=re.S):
            try:
                j = json.loads(block)
                calls.append({"name": j["name"], "args": j.get("arguments", {}), "type": "tool_call",
                              "id": "call_" + uuid.uuid4().hex[:8]})
            except (json.JSONDecodeError, KeyError, TypeError):
                pass   # an unreadable tool call stays in the text, where you can see it
        content = re.sub(r"<tool_call>.*?</tool_call>", "", text, flags=re.S).strip() if calls else text
        return ChatResult(generations=[ChatGeneration(message=AIMessage(content=content, tool_calls=calls))])


def load_chat_model(name=None):
    """Load the local chat model as a LangChain chat model. name: None -> MAIN_MODEL (env var L05_MODEL overrides)."""
    return LocalChatModel(model_name=name or os.environ.get("L05_MODEL") or MAIN_MODEL)


def show_prompt(width=110, max_lines=60):
    """Print the LAST text the model really received (after LangChain and the chat template), and its raw answer."""
    if LAST["prompt"] is None:
        print("(the model was not called on the last run - e.g. the gate answered \"not found\")")
        return
    lines = LAST["prompt"].splitlines()
    print(f"--- the text the model received ({len(LAST['prompt'])} characters, {len(lines)} lines) ---")
    for line in lines[:max_lines]:
        print(line if len(line) <= width else line[: width - 3] + "...")
    if len(lines) > max_lines:
        print(f"... ({len(lines) - max_lines} more lines)")
    print(f"--- the model's raw answer ({LAST['seconds']} s) ---\n{LAST['answer']}")


# ======================================================================================  the knowledge base
def _chunks(doc, size, overlap):
    """Fixed chunks of `size` words; each repeats the last `overlap` words of the one before (Lesson 2.4)."""
    parts = re.split(r"^## ", doc["text"], flags=re.M)
    text = " ".join(f"{p.strip().partition(chr(10))[0]}. {' '.join(p.strip().partition(chr(10))[2].split())}"
                    for p in parts if p.strip())
    w = text.split()
    step = size - overlap
    return [" ".join(w[i:i + size]) for i in range(0, max(1, len(w) - overlap), step)]


def kb_documents(docs=None, size=None, overlap=None):
    """The handbook -> LangChain Documents (one per chunk, configuration of Lesson 2.4). Each Document has
    page_content (the chunk text) and metadata (chunk_id, source_id, title, category, updated_at, version)."""
    docs = KB_DOCS if docs is None else docs
    size, overlap = size or KB_CONFIG["size"], overlap if overlap is not None else KB_CONFIG["overlap"]
    out = []
    for d in docs:
        for n, t in enumerate(_chunks(d, size, overlap), 1):
            cid = f"{d['source_id']}#{n:02d}"
            out.append(Document(page_content=t, id=cid, metadata={
                "chunk_id": cid, "source_id": d["source_id"], "title": d["title"], "category": d["category"],
                "updated_at": d["updated_at"], "version": d["version"]}))
    return out


def load_embeddings():
    """The embedding model of Lessons 2.3-2.4 as a LangChain embeddings object."""
    from langchain_huggingface import HuggingFaceEmbeddings
    t0 = time.time()
    emb = HuggingFaceEmbeddings(model_name=EMBED_MODEL, model_kwargs={"device": DEVICE})
    print(f"Embedding model {EMBED_MODEL} | loaded in {time.time() - t0:.0f} s")
    return emb


def build_vector_store(embeddings, documents=None, name="company-x-kb", path=STORE):
    """Rebuild the Lesson 2.4 KB as a LangChain Chroma vector store (cosine; same ids = no copies). Returns it."""
    import chromadb
    from chromadb.config import Settings
    from langchain_chroma import Chroma
    documents = kb_documents() if documents is None else documents
    client = chromadb.PersistentClient(path=path, settings=Settings(anonymized_telemetry=False))
    if name in [c.name for c in client.list_collections()]:
        client.delete_collection(name)
    store = Chroma(client=client, collection_name=name, embedding_function=embeddings,
                   collection_configuration={"hnsw": {"space": "cosine"}},
                   relevance_score_fn=lambda distance: 1 - distance)   # similarity = 1 - cosine distance
    store.add_documents(documents, ids=[d.id for d in documents])
    print(f"[OK] vector store {name!r}: {store._collection.count()} chunks (Lesson 2.4 configuration: fixed "
          f"{KB_CONFIG['size']} words, overlap {KB_CONFIG['overlap']})")
    return store


def show_docs(docs, width=90):
    """Print retrieved Documents: chunk id, (similarity if known), date, start of the text."""
    if not docs:
        print("  (no document above the threshold)")
    for i, d in enumerate(docs, 1):
        d, score = (d if isinstance(d, tuple) else (d, d.metadata.get("similarity")))
        s = f"{score:.2f}" if score is not None else "  - "
        text = d.page_content if len(d.page_content) <= width else d.page_content[: width - 3] + "..."
        print(f"  {i}  {d.metadata['chunk_id']:<14} sim {s}  {d.metadata['updated_at']}  {text}")


def format_docs(docs):
    """Retrieved Documents -> the context text for the prompt: one block per chunk, starting with its id in [ ]."""
    if not docs:
        return "(no document found)"
    return "\n\n".join(f"[{d.metadata['chunk_id']}] (updated {d.metadata['updated_at']}) {d.page_content}"
                       for d in docs)


def number_docs(docs):
    """Retrieved Documents -> the context text with SHORT labels [1], [2], [3] (small models copy long ids badly:
    measured, the model wrote "POT-RET-01" for "POL-RET-01"). Code maps the numbers back to chunk ids."""
    if not docs:
        return "(no document found)"
    return "\n\n".join(f"[{i}] (updated {d.metadata['updated_at']}) {d.page_content}" for i, d in enumerate(docs, 1))


NOT_FOUND = {"answer": "I do not have this information.", "source_ids": [], "enough_info": False}


def make_rag(retriever, generate, gate=True):
    """Wire the RAG chain: question -> retriever -> (gate) -> generate -> answer with chunk ids.
    generate: prompt | chat model | parser, returning an object with answer, sources (block numbers), enough_info.
    gate=True: if the retriever finds NO chunk above the threshold, code answers "not found" and the model is not
    called at all. Returns a LangChain Runnable: rag.invoke(question) -> {"question", "docs", "answer"}."""
    from langchain_core.runnables import RunnableLambda, RunnablePassthrough

    def answer(x):
        if gate and not x["docs"]:
            LAST.update(prompt=None, answer=None, seconds=None)   # the model was NOT called for this question
            return dict(NOT_FOUND)
        try:
            a = generate.invoke({"context": number_docs(x["docs"]), "question": x["question"]})
        except Exception as e:  # noqa: BLE001 - an unreadable answer is a generation error, not a crash
            print(f"  [NO] the answer could not be read as the schema: {type(e).__name__}")
            return None
        if a is None:   # with_structured_output returns None when the model wrote no (valid) tool call
            print("  [NO] the model returned no valid answer (no tool call that fits the schema)")
            return None
        nums = _value(a, "sources", []) or []
        ids = [x["docs"][n - 1].metadata["chunk_id"] for n in nums if isinstance(n, int) and 1 <= n <= len(x["docs"])]
        return {"answer": _value(a, "answer", ""), "source_ids": ids, "enough_info": _value(a, "enough_info")}

    return {"docs": retriever, "question": RunnablePassthrough()} | RunnablePassthrough.assign(
        answer=RunnableLambda(answer))


# ======================================================================================  checking a RAG answer
def _value(answer, key, default=None):
    if answer is None:
        return default
    if isinstance(answer, dict):
        return answer.get(key, default)
    return getattr(answer, key, default)


def check_rag(question, docs, answer, show=True):
    """Score one RAG result against the question's ground truth. Returns a dict:
    retrieval_ok - answerable: the correct source_id is among the retrieved chunks; out of scope: always True
                   (an out-of-scope question has no right document - judge the answer)
    enough_ok    - enough_info is True for an answerable question, False for an out-of-scope one
    cites_ok     - every cited id was really retrieved, and an answerable answer cites the correct document
    fact_ok      - the answer contains one of the expected facts (answerable questions only)
    error        - "none", "retrieval" (the right text never reached the model) or "generation" (it did, or
                   nothing should have been answered, but the answer is wrong)"""
    got_ids = [d.metadata["chunk_id"] for d in docs]
    got_sources = {d.metadata["source_id"] for d in docs}
    text = str(_value(answer, "answer", "") or "")
    cited = [str(c).strip().strip("[]") for c in (_value(answer, "source_ids", []) or [])]   # "[ID]" -> "ID"
    enough = _value(answer, "enough_info", None)
    oos = question["source_id"] is None
    r = {"qid": question["qid"], "question": question["question"], "correct_source": question["source_id"] or "none",
         "retrieved": " ".join(got_ids), "answer": text, "source_ids": " ".join(cited), "enough_info": enough}
    r["retrieval_ok"] = True if oos else question["source_id"] in got_sources
    r["enough_ok"] = (enough is False) if oos else (enough is True)
    real = all(c in got_ids for c in cited)
    r["cites_ok"] = real if oos else (real and any(c.split("#")[0] == question["source_id"] for c in cited))
    r["fact_ok"] = None if oos else any(f.lower() in text.lower() for f in question["facts"])
    if oos:
        ok = r["enough_ok"]
    else:
        ok = r["retrieval_ok"] and r["enough_ok"] and r["cites_ok"] and r["fact_ok"]
    r["ok"] = ok
    r["error"] = "none" if ok else ("retrieval" if not r["retrieval_ok"] else "generation")
    if show:
        mark = lambda v: "--" if v is None else ("OK" if v else "NO")  # noqa: E731
        print(f"{question['qid']} retrieval {mark(r['retrieval_ok'])} | enough_info {mark(r['enough_ok'])} | "
              f"sources {mark(r['cites_ok'])} | fact {mark(r['fact_ok'])}  -> error: {r['error']}")
    return r


def run_questions(rag, questions=None, show=True):
    """Run a RAG chain (or function) on every question. rag.invoke(question) / rag(question) must return
    {"docs": [...], "answer": ...}.
    Prints one line per question and the totals. Returns the list of check_rag rows."""
    questions = QUESTIONS if questions is None else questions
    rows = []
    for q in questions:
        t0 = time.time()
        out = rag.invoke(q["question"]) if hasattr(rag, "invoke") else rag(q["question"])
        r = check_rag(q, out["docs"], out["answer"], show=False)
        r["seconds"] = round(time.time() - t0, 1)
        rows.append(r)
        if show:
            short = r["answer"][:52] + ("..." if len(r["answer"]) > 52 else "")
            print(f"{q['qid']} {'OK ' if r['ok'] else 'NO '} error={r['error']:<10} enough={str(r['enough_info']):<5} "
                  f"src=[{r['source_ids']}]  {short}")
    if show:
        print(totals(rows))
    return rows


def totals(rows):
    n = len(rows)
    ok = sum(r["ok"] for r in rows)
    ret = sum(r["error"] == "retrieval" for r in rows)
    gen = sum(r["error"] == "generation" for r in rows)
    return f"{ok}/{n} correct | retrieval errors {ret} | generation errors {gen}"


def compare_runs(a, b, labels=("before", "after")):
    """Two run_questions results side by side, one line per question."""
    print(f"{'qid':<5}{labels[0]:>22}{labels[1]:>22}")
    for x, y in zip(a, b):
        cell = lambda r: ("OK " if r["ok"] else "NO ") + r["error"]  # noqa: E731
        print(f"{x['qid']:<5}{cell(x):>22}{cell(y):>22}")
    print(f"{'':<5}{sum(r['ok'] for r in a):>19}/{len(a)}{sum(r['ok'] for r in b):>19}/{len(b)}")


def show_rag(out, width=90):
    """Print a RAG result in reading order: first the retrieved chunks, then the answer."""
    print("Retrieved chunks:")
    show_docs(out["docs"], width=width)
    a = out["answer"]
    print("Answer:" if a else "Answer: (none - the model's reply could not be read)")
    if a:
        print(json.dumps(a, indent=2))


# ---------- Part 2: a ticket, checked against the ground truth of Lesson 2.3
def check_ticket(value, ticket_id):
    """Compare an extracted ticket with its ground truth (order_id, product, category, amount_vnd).
    Prints [OK] or one [NO] line per wrong field. Returns True if all four are right."""
    truth = TICKETS[ticket_id]["truth"]
    v = value.model_dump() if hasattr(value, "model_dump") else (value or {})
    wrong = []
    for f in ("order_id", "category", "amount_vnd"):
        if v.get(f) != truth[f]:
            wrong.append(f)
    got, want = (v.get("product") or "").lower(), (truth["product"] or "").lower()
    if not ((not got and not want) or (got and want and (got in want or want in got))):
        wrong.append("product")
    print(f"{ticket_id} " + ("[OK] order_id, product, category, amount_vnd" if not wrong else
                             " ".join(f"[NO] {f}: {json.dumps(v.get(f))} (truth {json.dumps(truth[f])})"
                                      for f in wrong)))
    return not wrong


# ======================================================================================  the hand-made version
def manual_rag(question, chat, store, k=3, threshold=0.35):
    """The same RAG WITHOUT LangChain chains (Lesson 2.4 style): query the collection, build the prompt text by
    hand, call the model, read the JSON with json.loads. Returns {"docs", "answer", "prompt", "raw"}."""
    col = store._collection
    res = col.query(query_texts=None, query_embeddings=[store._embedding_function.embed_query(question)],
                    n_results=k)
    docs = []
    for cid, text, meta, dist in zip(res["ids"][0], res["documents"][0], res["metadatas"][0], res["distances"][0]):
        if 1 - dist >= threshold:
            docs.append(Document(page_content=text, metadata=dict(meta, similarity=round(1 - dist, 3))))
    prompt = ("Answer the customer's question using ONLY the context. Each context block starts with its id in [ ].\n"
              'Reply with only a JSON object: {"answer": "...", "source_ids": ["..."], "enough_info": true}\n'
              "If the context does not answer the question, set enough_info to false, source_ids to [] and say so.\n\n"
              f"Context:\n{format_docs(docs)}\n\nQuestion: {question}")
    raw = chat.invoke([HumanMessage(prompt)]).content
    m = re.search(r"\{.*\}", raw, flags=re.S)
    try:
        answer = json.loads(m.group(0)) if m else None
    except json.JSONDecodeError:
        answer = None
    return {"docs": docs, "answer": answer, "prompt": prompt, "raw": raw}


# ======================================================================================  saving
def save_json(value, path):
    """Save any JSON value to a .json file (pretty, ASCII). Prints the file path."""
    p = Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(json.dumps(value, indent=2, ensure_ascii=True, default=str), encoding="ascii")
    print(f"[OK] saved -> {p}")
    return p


def save_results(rows, path="results/results.csv"):
    """Save a list of dicts to a CSV file (opens in Excel / Google Sheets). Prints the file path."""
    p = Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    keys = list(dict.fromkeys(k for r in rows for k in r))
    with p.open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=keys)
        w.writeheader()
        w.writerows(rows)
    print(f"[OK] saved {len(rows)} rows -> {p}")
    return p

print("[OK] Helpers loaded")

[OK] Helpers loaded


In [4]:
# Cell S2 - the chat model, the embedding model and the knowledge base of Lesson 2.4 (about 30 s)
# Little free memory? Use the smaller model instead:  chat = load_chat_model(SMALL_MODEL)
chat = load_chat_model()
embeddings = load_embeddings()
store = build_vector_store(embeddings)        # rebuilt from KB_DOCS: the checkpoint of Lesson 2.4

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Chat model Qwen/Qwen2.5-1.5B-Instruct | 1.54 billion parameters | cpu | loaded in 10 s


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding model sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 | loaded in 11 s
[OK] vector store 'company-x-kb': 29 chunks (Lesson 2.4 configuration: fixed 60 words, overlap 15)


In [5]:
# TODO: your full name
STUDENT_NAME = "Nguyễn Văn Bảo Phúc"

In [6]:
# Cell S3 - the chain of the demo: prompt (D01), retriever (D02), schema + chain (D03), with the gate
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field

rag_template = """Answer the customer's question using ONLY the context. The context blocks are numbered [1], [2], [3].
Rules:
- Use only facts from the context, never your own knowledge about shops.
- sources: the numbers of the blocks you used.
- If the context does not answer the question: enough_info false, sources [], answer "I do not have this information."
- If two blocks disagree, follow the block updated most recently.

Context:
{context}

Question: {question}"""
rag_prompt = ChatPromptTemplate.from_messages([("system", 'You are a support assistant at Company X, an online store for home appliances.'), ("human", rag_template)])
retriever = store.as_retriever(search_type="similarity_score_threshold",
                               search_kwargs={"k": 3, "score_threshold": 0.35})

class Answer(BaseModel):
    """The answer to the customer's question, based only on the context."""
    answer: str = Field(description="the answer in 1-2 sentences, using only facts from the context")
    sources: list[int] = Field(description="the numbers [1], [2], [3] of the context blocks used; [] if none")
    enough_info: bool = Field(description="true if the context answers the question; false if it does not")

generate = rag_prompt | chat.with_structured_output(Answer)
rag = make_rag(retriever, generate, gate=True)
print("[OK] chain ready")

[OK] chain ready


## Part A - Your 8 questions

**Today's job at Company X** (fictional online store for home appliances):

| Box | Today's answer |
|---|---|
| The problem | Support agents answer the same policy questions all day. A chat model alone does not know Company X's policies; the handbook (Lesson 2.4 KB) does not write answers. |
| Today's job | **RAG**: retrieve the right chunks, put them in the prompt, let the model write a short answer **with its sources** - or say "I do not have this information" |
| Tools | **LangChain**: messages, prompt templates, a chat model, a retriever, an output schema - joined into a chain |
| Pass criterion | each answer cites a chunk that really supports it; the 2 out-of-scope questions get `enough_info: false` |
| Human check | an agent reads the cited chunk before sending; the model's answer is a draft |

### A1 - Read the questions and predict

In [7]:
# Your 8 questions (run it, no changes needed)
for q in TASK_QUESTIONS:
    print(f"{q['qid']}  {q['kind']:<13} {str(q['source_id']):<11} {q['question']}")

T01  paraphrase    POL-RET-01  I opened the box and I don't want it anymore. Can I return it?
T02  two versions  POL-RET-01  Who pays the courier when I send an item back?
T03  product code  FAQ-AP-01   Is HF-2013 the right filter for my purifier?
T04  late section  POL-PAY-01  I paid twice for the same order. When do I get one payment back?
T06  paraphrase    SVC-INS-01  How much does it cost to install an air conditioner?
T08  late section  FAQ-AP-01   How loud is the bigger purifier at night?
T09  out of scope  None        Can I collect my order myself at your warehouse?
T10  out of scope  None        Do you buy back old phones?


In [8]:
# TODO A1 - BEFORE running: which questions will go wrong, and is it retrieval or generation? (2-3 sentences, IDs)
my_prediction = "..."

In [1]:
# Cell A2 - run the chain on your 8 questions and save the table (about 1 minute)
run_a = run_questions(rag, TASK_QUESTIONS)
saved_a = save_results(run_a, "results/part_a_answers.csv")

NameError: name 'run_questions' is not defined

## Part B - Change ONE setting
Choose **one**: `k` (1 or 5), a metadata `filter` (e.g. `{"category": "returns"}` for one question), one more rule
in the prompt, or `gate=False`. Keep everything else as in S3.

In [ ]:
# Cell B1 - my change
# TODO: describe your ONE change, then build rag_b with it (copy from S3 and change one thing).
my_change = "..."
rag_template = """Answer the customer's question using ONLY the context. The context blocks are numbered [1], [2], [3].
Rules:
- Use only facts from the context, never your own knowledge about shops.
- sources: the numbers of the blocks you used.
- If the context does not answer the question: enough_info false, sources [], answer "I do not have this information."
- If two blocks disagree, follow the block updated most recently.
Context:
{context}

Question: {question}"""
retriever_b = store.as_retriever(search_type="similarity_score_threshold",
                                 search_kwargs={"k": 3, "score_threshold": 0.35})
rag_b = make_rag(retriever_b, generate, gate= False)
run_b = run_questions(rag_b, TASK_QUESTIONS)

T01 OK  error=none       enough=True  src=[POL-RET-01#01]  Yes, you can return the item within 14 days of deliv...
T02 NO  error=generation enough=True  src=[POL-RET-01#02 POL-RET-01#04]  We ship the new item when the old one reaches our wa...
T03 OK  error=none       enough=True  src=[FAQ-AP-01#02]  Yes, HF-2013 is the correct filter for your AP-20 pu...
T04 OK  error=none       enough=True  src=[POL-PAY-01#03]  A double charge is refunded in 3-5 working days, dep...
T06 OK  error=none       enough=True  src=[SVC-INS-01#02]  It costs 500,000 VND for an air conditioner to be in...
T08 OK  error=none       enough=True  src=[FAQ-AP-01#02]  The AP-40 makes 28 dB at night.


No relevant docs were retrieved using the relevance score threshold 0.35


T09 NO  error=generation enough=True  src=[POL-ORD-01#01 POL-ORD-01#02]  No, you cannot collect your order yourself at our wa...
T10 NO  error=generation enough=True  src=[]  No, we don't currently offer phone buying-back servi...
5/8 correct | retrieval errors 0 | generation errors 3


In [ ]:
# Cell B2 - before vs after (run it, no changes needed)
compare_runs(run_a, run_b, ("S3 chain", "my change"))
saved_b = save_results([dict(r, run="before") for r in run_a] + [dict(r, run="after") for r in run_b],
                       "results/part_b_before_after.csv")

qid                S3 chain             my change
T01                 OK none               OK none
T02           NO generation         NO generation
T03                 OK none               OK none
T04                 OK none               OK none
T06                 OK none               OK none
T08                 OK none               OK none
T09           NO generation         NO generation
T10                 OK none         NO generation
                       6/8                  5/8
[OK] saved 16 rows -> results\part_b_before_after.csv


## Part C - The same question, with and without LangChain chains
`manual_rag` does the same job by hand (Lesson 2.4 style): query the collection, build the prompt text with an
f-string, call the model, `json.loads` the reply. Pick **one** of your questions.

In [ ]:
# TODO: pick one question (e.g. TASK_QUESTIONS[0]["question"])
my_question = "How loud is the bigger purifier at night?"

In [ ]:
# Cell C1 - hand-made vs LangChain on your question (run it, no changes needed)
hand = manual_rag(my_question, chat, store)
print("=== HAND-MADE: the prompt text ===\n" + hand["prompt"])
print("\n=== HAND-MADE: raw reply ===\n" + hand["raw"])
lc = rag.invoke(my_question)
print("\n=== LANGCHAIN: the text the model received ===")
show_prompt(max_lines=40)
print("\n=== LANGCHAIN: result ===")
show_rag(lc)

=== HAND-MADE: the prompt text ===
Answer the customer's question using ONLY the context. Each context block starts with its id in [ ].
Reply with only a JSON object: {"answer": "...", "source_ids": ["..."], "enough_info": true}
If the context does not answer the question, set enough_info to false, source_ids to [] and say so.

Context:
[FAQ-AP-01#03] (updated 2026-06-20) HF-2013 (390,000 VND); for the AP-40 it is HF-4013 (450,000 VND). Cleaning and noise. Wipe the outside and the sensor with a dry cloth once a month. In night mode the AP-20 makes 24 dB and the AP-40 makes 28 dB, quieter than a whisper.

[FAQ-AP-01#02] (updated 2026-06-20) the air quality: blue is good, yellow is fair and red is bad. In automatic mode the fan speed follows the light. Filters. Each purifier comes with one filter. Filters are not washable: replace them every 6 months. The spare filter for the AP-20 is HF-2013 (390,000 VND); for the AP-40 it is HF-4013 (450,000 VND). Cleaning and noise. Wipe

Question: Ho

In [ ]:
# TODO C2 - 3-5 sentences: what is the same, what is different (prompt, parsing, sources), what LangChain hides
my_comparison = "..."

## Part D - Read the errors
### D1 - Two analyses
One **retrieval** error and one **generation** error from Part A (if there is no retrieval error, say why and
analyse two generation errors). For each: the question ID, the retrieved chunk ids, the answer, and the evidence
(quote the chunk or the answer).

In [1]:
# Cell D1 - the questions of Part A that are not OK: chunks first, then the answer (run it)
for r in run_a:
    if not r["ok"]:
        q = next(x for x in TASK_QUESTIONS if x["qid"] == r["qid"])
        print(f"\n=== {q['qid']} {q['question']}  (correct: {q['source_id']}, expected: {q['facts']}) -> {r['error']}")
        show_rag(rag.invoke(q["question"]))

NameError: name 'run_a' is not defined

In [ ]:
# TODO D1: two entries "question ID": "retrieval/generation - evidence from the output above"
error_analysis = {
    "...": "...",
    "...": "...",
}

### D2 - Does the source support the claim?
Pick **three** answers of Part A that are marked OK. For each, quote the sentence of the **cited chunk** that
supports the answer - or write "not supported" if you cannot find it.

In [ ]:
# TODO D2: three entries "question ID": "chunk id: the quoted sentence" (or "not supported")
support_check = {
    "...": "...",
    "...": "...",
    "...": "...",
}

In [ ]:
# Final check before you submit (no changes needed)
def _ok(test):
    try:
        return bool(test())
    except Exception:  # noqa: BLE001 - that part is not finished yet
        return False

checks = {
    "full name": _ok(lambda: STUDENT_NAME.strip()),
    "A1 prediction": _ok(lambda: "..." not in my_prediction and len(my_prediction.split()) >= 10),
    "A2 eight answers saved": _ok(lambda: len(run_a) == 8 and saved_a.exists()),
    "B1 one change described": _ok(lambda: "..." not in my_change and len(my_change.split()) >= 2),
    "B2 before/after saved": _ok(lambda: saved_b.exists()),
    "C hand-made vs LangChain": _ok(lambda: "..." not in my_question and "..." not in my_comparison
                                    and len(my_comparison.split()) >= 30),
    "D1 two analyses": _ok(lambda: len(error_analysis) == 2 and "..." not in json.dumps(error_analysis)),
    "D2 three support checks": _ok(lambda: len(support_check) == 3 and "..." not in json.dumps(support_check)),
}
print()
for k, v in checks.items():
    print(("[OK] " if v else "[NO] ") + k)
print("Ready to submit" if all(checks.values()) else "Some items are [NO] - review them before submitting")
print("Also check by reading: your analyses quote chunks and answers from YOUR outputs.")